# Notebook Prerequisites

- **Kernel / Python:** Use the project's virtual environment (see `pyproject.toml`).
- **Required packages:** `pandas`, `scikit-learn`, `statsmodels`.
- **Data:** Processed datasets live at `data/processed`. Run `02_feature_eng_encoding.ipynb` first if files are missing.

In [2]:
# ================================================
# 1. Imports
# ================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [3]:
from pathlib import Path
# ================================================
# 2. Load datasets (train + eval)
# ================================================
DATA_DIR = Path("../data/processed")
train_path = DATA_DIR / "feature_engineered_train.csv"
eval_path = DATA_DIR / "feature_engineered_eval.csv"

if not train_path.exists() or not eval_path.exists():
    raise FileNotFoundError(
        f"Expected processed files at {DATA_DIR.resolve()} - run `02_feature_eng_encoding.ipynb` first to create them."
    )

train_df = pd.read_csv(train_path)
eval_df  = pd.read_csv(eval_path)

In [4]:
'''
# ================================================
# 3. Drop high VIF features (both train + eval)
# ================================================
high_vif_features = [
    "median_sale_price" #highest correlation to 'price' => data leakage
]
train_df.drop(columns=high_vif_features, inplace=True)
eval_df.drop(columns=high_vif_features, inplace=True)
'''

'\n# ================================================\n# 3. Drop high VIF features (both train + eval)\n# ================================================\nhigh_vif_features = [\n    "median_sale_price" #highest correlation to \'price\' => data leakage\n]\ntrain_df.drop(columns=high_vif_features, inplace=True)\neval_df.drop(columns=high_vif_features, inplace=True)\n'

In [5]:
# ================================================
# 4. Define target & features
# ================================================
target = "price"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_eval = eval_df.drop(columns=[target])
y_eval = eval_df[target]

# Align eval to train features if there is a mismatch
missing_cols = [c for c in X_train.columns if c not in X_eval.columns]
if missing_cols:
    print(f"⚠️ Eval set is missing {len(missing_cols)} columns compared to train. Filling missing columns with 0 for demo purposes.")
    for c in missing_cols:
        X_eval[c] = 0

# Re-order columns to match
X_eval = X_eval.reindex(columns=X_train.columns)

In [6]:
# ================================================
# 5. Standardization (fit on train, transform eval)
# ================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled  = scaler.transform(X_eval)

In [7]:
# ================================================
# 6. Train & Evaluate Models
# ================================================

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_eval_scaled)

print("Linear Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lr))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lr)))
print(" R²:", r2_score(y_eval, y_pred_lr))

Linear Regression:
 MAE: 53811.9381340082
 RMSE: 121336.13469295994
 R²: 0.8862267031700114


In [8]:
# --- Ridge Regression ---
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_eval_scaled)

print("\nRidge Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_ridge))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_ridge)))
print(" R²:", r2_score(y_eval, y_pred_ridge))


Ridge Regression:
 MAE: 53811.11466043909
 RMSE: 121338.02551498664
 R²: 0.8862231572068501


In [9]:
# --- Lasso Regression ---
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_eval_scaled)

print("\nLasso Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lasso))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lasso)))
print(" R²:", r2_score(y_eval, y_pred_lasso))


Lasso Regression:
 MAE: 54117.42607123022
 RMSE: 121604.47823437207
 R²: 0.8857229111303108


c:\Users\vigne\Desktop\Rml\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.114e+15, tolerance: 5.209e+12
  model = cd_fast.enet_coordinate_descent(


In [10]:
# --- ElasticNet ---
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_eval_scaled)

print("\nElasticNet Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_elastic))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))
print(" R²:", r2_score(y_eval, y_pred_elastic))


ElasticNet Regression:
 MAE: 54234.24932061455
 RMSE: 122295.84870428534
 R²: 0.8844197946433395
